# AutoGrasp V2 — collect ANY object (runnable path)

Reuses V1's hardware-proven grasp cell (the 97%-policy machinery) with **one verified change**:
the shape-gated **ladder** drives the grasp angle for non-box objects (boxes keep V1's path).
Object selection = `ACTIVE_OVERRIDE`; V1 already segments any object (SAM3) + gets DeliGrasp
force per object. `wrist_fz` records itself once the FT node is up (recorder already subscribes).

**Order:** ①Config ②Imports ③Self-test(no robot) ④Perception helpers ⑤Dry-run(no motion) ⑥Bringup ⑦Collect.
V1 frozen at `v1-pipeline`. This is branch `v2-dev`.


## 1 · Config


In [ ]:
OBJECT_NAME  = "red cube"     # any object; SAM3 concept prompt + GraspMemory key
DELICATE     = False             # rigid (3D-printed) objects can use False
PART_PROMPT  = None             # "handle" etc. for part-directed; None = whole object

# Angle source. Start FALSE = cross-check only: the ladder angle is LOGGED next to
# minAreaRect every grasp (a live bake-off) but V1 still drives. Once the logged
# ladder angles look right on your objects, set True to let the ladder DRIVE
# (non-box shapes only; boxes always use V1's proven path).
LADDER_DRIVES_ANGLE = False

print(f"object={OBJECT_NAME!r} delicate={DELICATE} part={PART_PROMPT} "
      f"ladder_drives={LADDER_DRIVES_ANGLE}")
print("TIP: specific nouns segment best (measured): 'ripe strawberry' > 'object'.")


## 2 · Imports + 3 · Offline self-test (no robot — must say READY)


In [ ]:
import sys, os, json, tempfile, base64, socket as _sock, numpy as np, cv2
sys.path.insert(0, os.path.expanduser("~/magpie_control/src"))
sys.path.insert(0, os.path.expanduser("~/magpie_control/scripts"))
from magpie_control.v2.grasp_planner  import plan, priors_from_memory
from magpie_control.v2.grasp_ladder   import propose
import importlib.util as _u
_sp=_u.spec_from_file_location("v2st", os.path.expanduser("~/magpie_control/scripts/v2_selftest.py"))
_st=_u.module_from_spec(_sp); _sp.loader.exec_module(_st)
assert _st.run(), "offline pipeline NOT ready"
print("\n^ decision pipeline sound.")


## 4 · Perception helpers (real SAM3, tested)


In [ ]:
SAM3_SOCK = "/tmp/sam3.sock"
def sam3_query(img_rgb, query, sock_path=SAM3_SOCK):
    tmp = tempfile.mktemp(suffix=".jpg"); cv2.imwrite(tmp, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    try:
        with _sock.socket(_sock.AF_UNIX, _sock.SOCK_STREAM) as s:
            s.connect(sock_path); s.sendall((json.dumps({"image": tmp, "query": query})+"\n").encode())
            raw=b""
            while True:
                c=s.recv(65536)
                if not c: break
                raw+=c
        d=json.loads(raw.decode().strip())
        if "error" in d: raise RuntimeError(d["error"])
        boxes=np.array(d["boxes"],float); scores=np.array(d["scores"],float); mask=None
        if d.get("mask_b64") and len(boxes)>0:
            h,w=d["mask_shape"]
            mask=np.frombuffer(base64.b64decode(d["mask_b64"]),np.uint8).reshape(h,w).astype(bool)
        return boxes, scores, mask
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

def best_mask(img_rgb, prompt):
    boxes, scores, mask = sam3_query(img_rgb, prompt)
    if mask is None or mask.sum() < 200:
        return None, 0.0
    return mask.astype(np.uint8)*255, float(scores.max() if len(scores) else 0.0)

def depth_scale_mm_per_px(depth, mask, fx):
    """mm-per-pixel at the object plane from median depth under the mask."""
    z = depth[mask.astype(bool)].astype(float); z = z[z>0]
    if z.size < 20: return None
    return float(np.median(z) / fx)      # depth(mm) / focal(px)

def perceive_and_plan(node, gm):
    prompt = PART_PROMPT or OBJECT_NAME
    mask, score = best_mask(node.color, prompt)
    if mask is None:
        print(f"  SAM3 found no '{prompt}' — reword the prompt (specific nouns work best)"); return None
    fx = node.caminfo.k[0]
    mpp = depth_scale_mm_per_px(node.depth, mask, fx)
    if mpp is None:
        print("  depth too sparse under mask — nudge closer / re-scan"); return None
    fpri, wpri = priors_from_memory(gm, OBJECT_NAME)
    p = plan(mask, mpp, object_name=OBJECT_NAME, force_prior_n=fpri, width_prior_mm=wpri,
             max_width_mm=GRIPPER_MAX_MM, delicate=DELICATE)
    print(f"  SAM3 score={score:.2f} | [{p.method}] yaw={p.grasp_yaw_deg:.0f} "
          f"width={p.width_mm}mm seed={p.seed_force_n}N band={p.aperture_band_mm} prior={p.have_prior}")
    return p, mask, mpp



## 5 · PERCEPTION DRY-RUN — no motion  ⚠️ camera only
Run FIRST on every new object. Overlays the planned grasp; arm does NOT move.


In [ ]:
def perception_dryrun(node, gm):
    node.spin(6)
    out = perceive_and_plan(node, gm)
    if out is None:
        print("  DRY-RUN: no plan — fix perception before collecting"); return
    p, mask, mpp = out
    vis = node.color.copy()
    cnts,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cnts, -1, (0,255,0), 2)
    cx,cy = map(int, p.center_px); a=np.radians(p.grasp_yaw_deg); L=40
    cv2.line(vis,(int(cx-L*np.cos(a)),int(cy-L*np.sin(a))),(int(cx+L*np.cos(a)),int(cy+L*np.sin(a))),(255,0,0),3)
    cv2.circle(vis,(cx,cy),6,(0,0,255),-1)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7,5)); plt.imshow(vis); plt.title(f"{OBJECT_NAME}: {p.method} yaw={p.grasp_yaw_deg:.0f} seed={p.seed_force_n}N"); plt.axis("off"); plt.show()
    print("  green=mask  blue=grasp line  red=grasp center. Looks right? -> collect.")
# perception_dryrun(node, gm)



## 6 · Bringup  ⚠️ hardware
Terminal: `bash ~/magpie_control/scripts/bringup_gripper_ft.sh`. Build the V1 node
(v1_collect c01→c03→c04→a8581dd2), then set the object + **verify FT publishes** (else wrist_fz is 0):


In [ ]:
from grasp_memory import GraspMemory
# node = <build via v1_collect cells c01,c03,c04 — drivers/cameras/services>
gm = GraspMemory(os.path.expanduser("~/magpie_control/data/grasp_log"))

ACTIVE_OVERRIDE = OBJECT_NAME        # <-- object-agnostic: V1 grasp cell reads this

# VERIFY the FT sensor is publishing, or wrist_fz records as 0 (the V1 bug was operational):
import subprocess
_ft = subprocess.run("source /opt/ros/humble/setup.bash && "
                     "timeout 5 ros2 topic echo /ft_sensor/wrench --once",
                     shell=True, capture_output=True, text=True, executable="/bin/bash")
print("FT publishing:" , "OK" if "force" in _ft.stdout else "*** SILENT — start ft_sensor_node ***")
print(f"ACTIVE_OVERRIDE={ACTIVE_OVERRIDE!r}. Run cell 5 dry-run on your object, then cell 7.")


## 7 · Collect — V1 grasp cell + the ladder patch  ⚠️ hardware
Loads V1's proven grasp cell, injects the ladder (verified anchor + convention), runs it.
Repeat for each placement toward your episode target. Ladder angle is logged every grasp.


In [ ]:
# ensure the object is set (cell 6 may have been skipped) + sanity-check
# that the V1 setup cells (node, gc, gm, home_pick...) ran in THIS kernel.
ACTIVE_OVERRIDE = OBJECT_NAME
for _need in ["node", "gc", "gm"]:
    assert _need in dir(), f"run the v1_collect setup cells first — {_need!r} not defined"

# Load V1's grasp cell and apply the ONE verified V2 patch.
_v1 = json.load(open(os.path.expanduser("~/magpie_control/notebooks/v1_collect.ipynb")))
_GRASP = next("".join(c["source"]) for c in _v1["cells"]
              if c.get("id","").startswith("17ea9af9"))

_ANCHOR = "                _flat_mask = (90. - float(_mrect)) % 90."
assert _GRASP.count(_ANCHOR) == 1, "V1 grasp cell changed — re-verify the ladder anchor before running"

_LADDER = _ANCHOR + """
                # ── V2 ladder cross-check / drive (branch v2-dev) ──────────
                try:
                    from magpie_control.v2.grasp_ladder import propose as _v2_propose
                    _v2p    = _v2_propose((mask_use > 0).astype('uint8') * 255)
                    _v2flat = (90. - _v2p.grasp_yaw_deg) % 90.
                    _v2dev  = min((_v2flat - _flat_mask) % 90, (_flat_mask - _v2flat) % 90)
                    print(f'  [ladder] {_v2p.method}: flat={_v2flat:.0f} vs minAreaRect={_flat_mask:.0f} (dev {_v2dev:.0f} deg)')
                    globals()['_v2_ladder_method'] = _v2p.method
                    globals()['_v2_ladder_flat']   = float(_v2flat)
                    if globals().get('LADDER_DRIVES_ANGLE') and _v2p.method != 'minAreaRect':
                        print(f'  [ladder] DRIVING (non-box): _flat_mask {_flat_mask:.0f} -> {_v2flat:.0f}')
                        _flat_mask = _v2flat
                except Exception as _v2e:
                    print(f'  [ladder] cross-check skipped: {_v2e}')"""

_GRASP_PATCHED = _GRASP.replace(_ANCHOR, _LADDER, 1)
print("patched V1 grasp cell (ladder injected). running one grasp on:", ACTIVE_OVERRIDE)
get_ipython().run_cell(_GRASP_PATCHED)


## Notes
- **New object:** edit cell 1 (`OBJECT_NAME`, `DELICATE`), run cell 5 dry-run, then cell 7.
- **Angle:** starts in cross-check mode — collect a few, read the `[ladder]` dev vs minAreaRect,
  then set `LADDER_DRIVES_ANGLE=True` to let it drive non-box shapes. Boxes never change.
- **wrist_fz** records automatically once `ft_sensor_node` publishes (cell 6 verifies).
- **Delicacy** is DeliGrasp (already per-object in the V1 cell) on the rubber grippers — no hardware change.
- The only V2 code path here is the verified ladder patch; everything else is the 97%-proven V1 cell.
